In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1) Aspect listelerinizi kod içinde tanımlayın ---
aspects = {
    # 1. Seed Aspect (domain‐based)
    "room": ["room","floor","bedroom","bathroom","kitchen","balcony","bed","apartment","desk","hall","laundry","sofa","basement","spacious"],
    "service": ["service","staff","customer","maintenance","internet","support","quality","cleanliness","reliable"],
    "location": ["location","proximity","area","destination","vicinity","distance","close","parking"],
    "price": ["price","cost","value","discount","affordable","expensive","pay","worth"],
    "food": ["food","meal","restaurant","breakfast","lunch","dinner","soup","pizza","bread","coffee","dessert"],

    # 2. Discovered Aspects
    "Beach & Water": ["beach","sea","water","pool"],
    "Accommodation & Facilities": ["bathroom","shower","apartment","hotel","location"],
    "Service & Social Experience": ["staff","place","family","people","night","problem","river","price"],
    "Leisure & Meals": ["breakfast","beach","holiday"],
}

aspect_names = list(aspects.keys())

# --- 2) Gerekli veriyi yükleyin ---
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
df = df.dropna(subset=["processed_final_review"]).reset_index(drop=True)

# --- 3) ABSA modelini yükleyin ---
MODEL = "yangheng/deberta-v3-base-absa-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# --- 4) Aspect‐Sentiment tahmin fonksiyonu ---
def predict_aspect_sentiment(review: str, aspect: str) -> str:
    """
    review: yorum metni
    aspect: aspect adı (örn. "room" veya "Beach & Water")
    döndürür: 'positive' | 'negative' | 'neutral'
    """
    # modelin ABSA formatı: [CLS] review [SEP] aspect [SEP]
    inputs = tokenizer(review, aspect, return_tensors="pt",
                       truncation=True, max_length=256).to(device)
    out    = model(**inputs)
    scores = F.softmax(out.logits[0], dim=-1)  # [neg, neu, pos]
    label_id = torch.argmax(scores).item()
    return model.config.id2label[label_id].lower()

# --- 5) Her yorum × her aspect için tahmin ---
records = []
for idx, row in df.iterrows():
    review = row["processed_final_review"]
    hotel  = row.get("hotel_name", "")
    for aspect in aspect_names:
        sentiment = predict_aspect_sentiment(review, aspect)
        records.append({
            "hotel_name": hotel,
            "review_idx": idx,
            "aspect": aspect,
            "sentiment": sentiment
        })

absa_df = pd.DataFrame(records)
absa_df.to_csv("all_reviews_aspect_sentiments1.csv", index=False, encoding="utf-8-sig")

# --- 6) Otel × Aspect bazında özet (pozitif oran) ---
summary = (
    absa_df
      .assign(is_positive = lambda d: d.sentiment == "positive")
      .groupby(["hotel_name","aspect"])["is_positive"]
      .mean()
      .reset_index()
      .rename(columns={"is_positive":"pos_ratio"})
)
summary.to_csv("hotel_aspect_summary1.csv", index=False, encoding="utf-8-sig")

print("✅ ABSA tamamlandı.")
print("  • all_reviews_aspect_sentiments1.csv → her yorum‐aspect için sentiment")
print("  • hotel_aspect_summary1.csv         → otel‐aspect bazında pozitif oran")


C:\Users\catsu\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✅ ABSA tamamlandı.
  • all_reviews_aspect_sentiments1.csv → her yorum‐aspect için sentiment
  • hotel_aspect_summary1.csv         → otel‐aspect bazında pozitif oran


In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1) Aspect listelerinizi kod içinde tanımlayın ---
aspects = {
    # 1. Seed Aspect (domain‐based)
    "room": ["room","floor","bedroom","bathroom","kitchen","balcony","bed","apartment","desk","hall","laundry","sofa","basement","spacious"],
    "service": ["service","staff","customer","maintenance","internet","support","quality","cleanliness","reliable"],
    "location": ["location","proximity","area","destination","vicinity","distance","close","parking"],
    "price": ["price","cost","value","discount","affordable","expensive","pay","worth"],
    "food": ["food","meal","restaurant","breakfast","lunch","dinner","soup","pizza","bread","coffee","dessert"],

    # 2. Discovered Aspects
    "Beach & Water": ["beach","sea","water","pool"],
    "Accommodation & Facilities": ["bathroom","shower","apartment","hotel","location"],
    "Service & Social Experience": ["staff","place","family","people","night","problem","river","price"],
    "Leisure & Meals": ["breakfast","beach","holiday"],
}

aspect_names = list(aspects.keys())

# --- 2) Gerekli veriyi yükleyin ---
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")
df = df.dropna(subset=["processed_final_review"]).reset_index(drop=True)

# --- 3) ABSA modelini yükleyin ---
MODEL = "yangheng/deberta-v3-base-absa-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL)
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# --- 4) Aspect‐Sentiment tahmin fonksiyonu ---
def predict_aspect_sentiment(review: str, aspect: str) -> str:
    """
    review: yorum metni
    aspect: aspect adı (örn. "room" veya "Beach & Water")
    döndürür: 'positive' | 'negative' | 'neutral'
    """
    # modelin ABSA formatı: [CLS] review [SEP] aspect [SEP]
    inputs = tokenizer(review, aspect, return_tensors="pt",
                       truncation=True, max_length=256).to(device)
    out    = model(**inputs)
    scores = F.softmax(out.logits[0], dim=-1)  # [neg, neu, pos]
    label_id = torch.argmax(scores).item()
    return model.config.id2label[label_id].lower()

# --- 5) Her yorum × her aspect için tahmin ---
records = []
for idx, row in df.iterrows():
    review = row["processed_final_review"]
    hotel  = row.get("hotel_name", "")
    for aspect in aspect_names:
        sentiment = predict_aspect_sentiment(review, aspect)
        records.append({
            "hotel_name": hotel,
            "review_idx": idx,
            "aspect": aspect,
            "sentiment": sentiment
        })

absa_df = pd.DataFrame(records)
absa_df.to_csv("all_reviews_aspect_sentiments1.csv", index=False, encoding="utf-8-sig")

# --- 6) Otel × Aspect bazında özet (pozitif oran) ---
summary = (
    absa_df
      .assign(is_positive = lambda d: d.sentiment == "positive")
      .groupby(["hotel_name","aspect"])["is_positive"]
      .mean()
      .reset_index()
      .rename(columns={"is_positive":"pos_ratio"})
)
summary.to_csv("hotel_aspect_summary1.csv", index=False, encoding="utf-8-sig")

print("✅ ABSA tamamlandı.")
print("  • all_reviews_aspect_sentiments1.csv → her yorum‐aspect için sentiment")
print("  • hotel_aspect_summary1.csv         → otel‐aspect bazında pozitif oran")
